**Import modules**

In [1]:
import torch
import torch
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.models.mnist_model import MNISTModel
from src.data.mnist import get_mnist_loaders

from src.data.cifar import get_cifar_loaders
from src.models.cifar_model import CIFARModel

from src.train.train import train_model

from src.utils.seed import set_seed

set_seed(0)

## Evaluation metrics

##### The following metrics are used in the experiments:

**1.Clean Accuracy** 

The proportion of objects correctly classified by the unperturbed model

$clean\_acc = \frac{\text{\# correct predictions}}{\text{total samples}}$

**2.Robust Accuracy (PGD)** 

Evaluates the resilience of a model to PGD attacks and is defined as

$robust\_acc = 1 - \frac{\text{\# successful attacks}}{\text{total samples}}$

The metric reflects the proportion of inputs for which the attack failed to change the model's prediction.

**3.Certification Rate (SDP)** 

The proportion of inputs for which it was possible to prove stability using SDP

$cert\_rate = \frac{\text{\# certified samples}}{\text{total samples}}$

In this experiment, all metrics are calculated using the same set of inputs, which allows:

- direct comparison of empirical stability (PGD)
- guaranteed stability (SDP)

## MNIST

### 1. Baseline Experiment 

**Model Setup**

- **Architecture:** 784 → 4 → 10 + ReLU
- **Epochs:** 3
- **Accuracy:** 0.8449

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 5e-4}
- Samples: 3
- PGD: 10 steps, $\alpha = \epsilon / 5$
- Verification: top-2 classes

**Results:**

- EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc=1.00
- EPS=0.00050 | clean_acc=1.00 | cert_rate=1.00 | robust_acc=1.00

**Conclusion**

For small values ​​of $\epsilon$, the following is observed:

- complete agreement between all metrics;
- no detected attacks;
- complete certifiability.

This is explained by the fact that:

- the perturbations are too small to change the prediction;
- even a weak model remains locally stable.

**For small values ​​of $\epsilon$, the model exhibits high local stability, which is confirmed by both the empirical PGD attack and the formal SDP certification.**

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainloader, testloader = get_mnist_loaders()

model = MNISTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_model(
    model=model,
    trainloader=trainloader,
    testloader=testloader,
    optimizer=optimizer,
    device=device,
    num_epochs=3,
    patience=5,
    save_path="best_mnist_model.pth"
)

Starting training...
Epoch 1/3, Test Accuracy: 0.7885
Epoch 2/3, Test Accuracy: 0.8258
Epoch 3/3, Test Accuracy: 0.8449
Best accuracy: 0.8449
Model saved to: best_mnist_model.pth


In [3]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=870.9s

EPS=0.00050 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=875.0s


### 2. Upgrade model (8 neurons)

**Model Setup**

- **Architecture:** 784 → 8 → 10 + ReLU
- **Epochs:** 50 (early stop at 15th)
- **Accuracy:** 0.9265

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 5e-4, 1e-3, 1e-2, 5e-2}
- Samples: 20
- PGD: 25 steps, $\alpha = \epsilon / 5$
- Verification: top-2 classes

**Results:**

- EPS=0.00010 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6544.0s
- EPS=0.00050 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6538.0s
- EPS=0.00100 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6536.6s
- EPS=0.01000 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6538.9s
- EPS=0.05000 | clean_acc=0.95 | cert_rate=0.80 | robust_acc(PGD)=0.80 | time=6387.8s

**Conclusion**

As $\epsilon$ increases, robustness begins to decrease. PGD begins to detect successful attacks. SDP detects a decrease in the region of guaranteed robustness.

The match between cert_rate ≈ robust_acc means:

- consistency between the empirical and formal estimates;
- absence of obvious contradictions between the methods.

**As $\epsilon$ increases, the expected decrease in model stability is observed. However, PGD and SDP yield consistent estimates, confirming the validity of the analytical methods used.**

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainloader, testloader = get_mnist_loaders()

model = MNISTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_model(
    model=model,
    trainloader=trainloader,
    testloader=testloader,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    patience=5,
    save_path="best_mnist_model.pth"
)

Starting training...
Epoch 1/50, Test Accuracy: 0.9015
Epoch 2/50, Test Accuracy: 0.9085
Epoch 3/50, Test Accuracy: 0.9161
Epoch 4/50, Test Accuracy: 0.9167
Epoch 5/50, Test Accuracy: 0.9194
Epoch 6/50, Test Accuracy: 0.9223
Epoch 7/50, Test Accuracy: 0.9232
Epoch 8/50, Test Accuracy: 0.9244
Epoch 9/50, Test Accuracy: 0.9229
Epoch 10/50, Test Accuracy: 0.9265
Epoch 11/50, Test Accuracy: 0.9264
Epoch 12/50, Test Accuracy: 0.9256
Epoch 13/50, Test Accuracy: 0.9246
Epoch 14/50, Test Accuracy: 0.9257
Epoch 15/50, Test Accuracy: 0.9256
Early stopping at epoch 15
Best accuracy: 0.9265
Model saved to: best_mnist_model.pth


In [3]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00010 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6544.0s

EPS=0.00050 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6538.0s

EPS=0.00100 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6536.6s

EPS=0.01000 | clean_acc=0.95 | cert_rate=0.95 | robust_acc(PGD)=0.95 | time=6538.9s


/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(



EPS=0.05000 | clean_acc=0.95 | cert_rate=0.80 | robust_acc(PGD)=0.80 | time=6387.8s


### 3. PGD ​​Enhanced Attack

**Model Setup**

- **Architecture:** 784 → 8 → 10 + ReLU
- **Epochs:** 50 (early stop at 15th)
- **Accuracy:** 0.9265

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 1e-2, 1e-1}
- Samples: 5
- PGD: 50 steps, $\alpha = \epsilon / 10$
- Verification: top-2 classes

**Results:**

- EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=1661.9s
- EPS=0.01000 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=1668.1s
- EPS=0.10000 | clean_acc=1.00 | cert_rate=0.20 | robust_acc(PGD)=0.20 | time=1004.5s

**Conclusion**

- Increasing the number of steps has no effect on small $\epsilon$;
- At $\epsilon$ = 0.1 the attack becomes effective.

PGD was already quite strong the model does indeed lose stability at large radii

**Strengthening the attack does not lead to significant changes in the results at small $\epsilon$, indicating that the initial attack is sufficiently strong. At large ε, a sharp drop in stability is observed.**

In [2]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=1661.9s

EPS=0.01000 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=1668.1s

EPS=0.10000 | clean_acc=1.00 | cert_rate=0.20 | robust_acc(PGD)=0.20 | time=1004.5s


### 4. Checking all classes

**Model Setup**

- **Architecture:** 784 → 8 → 10 + ReLU
- **Epochs:** 50 (early stop at 15th)
- **Accuracy:** 0.9265

**Attack and verification parameters**

- $\epsilon \in$ {1e-2, 1e-1}
- Samples: 2
- PGD: 50 steps, $\alpha = \epsilon / 10$
- Verification: full classes

**Results:**

- EPS=0.01000 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=3005.4s
- EPS=0.10000 | clean_acc=1.00 | cert_rate=0.00 | robust_acc(PGD)=0.00 | time=1172.1s

**Conclusion**

- For large $\epsilon$, the model is completely vulnerable;
- SDP correctly captures this.

! The results are consistent with the top-k approach.

**A full check across all classes confirms the correctness of using the top-k strategy, which significantly speeds up calculations without losing the quality of the estimate.**

In [2]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.01000 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=3005.4s

EPS=0.10000 | clean_acc=1.00 | cert_rate=0.00 | robust_acc(PGD)=0.00 | time=1172.1s


### 5. The biggest pgd attack

**Model Setup**

- **Architecture:** 784 → 8 → 10 + ReLU
- **Epochs:** 50 (early stop at 19th)
- **Accuracy:** 0.9265

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 1e-1}
- Samples: 2
- PGD: 200 steps, $\alpha = \epsilon / 10$
- Verification: full classes

**Results:**

- EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=3009.9s
- EPS=0.10000 | clean_acc=1.00 | cert_rate=0.00 | robust_acc(PGD)=0.00 | time=1165.6s

**Conclusion**

Increasing the number of PGD steps did not change the results.

This means that:
- The attack had already reached maximum efficiency;
- A local maximum had been found earlier.

**Increasing the number of PGD iterations to 200 does not change the results, indicating that the attack used is already strong enough and achieves a stationary solution.**

In [2]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00010 | clean_acc=1.00 | cert_rate=1.00 | robust_acc(PGD)=1.00 | time=3009.9s

EPS=0.10000 | clean_acc=1.00 | cert_rate=0.00 | robust_acc(PGD)=0.00 | time=1165.6s


### 6. Final model (16 neurons)

**Model Setup**

- **Architecture:** 784 → 16 → 10 + ReLU
- **Epochs:** 50 (early stop at 19th)
- **Accuracy:** 0.9542

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 5e-4, 1e-3, 1e-2, 5e-2}
- Samples: 12
- PGD: 50 steps, $\alpha = \epsilon / 10$
- Verification: top-2 classes

**Results:**

- EPS=0.00010 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4844.7s
- EPS=0.00050 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4843.5s
- EPS=0.00100 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4840.8s
- EPS=0.01000 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4841.5s
- EPS=0.05000 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.83 | time=4862.8s

**Conclusion**

- Increasing the model size improves quality;
- Robustness at small $\epsilon$ remains high;

At large $\epsilon$ PGD finds more attacks.

! The results are consistent with the top-k approach.

**The more powerful model exhibits better accuracy and more robust behavior. The difference between cert_rate and robust_acc for large $\epsilon$ reflects the difference between the empirical and guaranteed robustness estimates.**

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainloader, testloader = get_mnist_loaders()

model = MNISTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_model(
    model=model,
    trainloader=trainloader,
    testloader=testloader,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    patience=5,
    save_path="best_mnist_model.pth"
)

Starting training...
Epoch 1/50, Test Accuracy: 0.9175
Epoch 2/50, Test Accuracy: 0.9328
Epoch 3/50, Test Accuracy: 0.9422
Epoch 4/50, Test Accuracy: 0.9414
Epoch 5/50, Test Accuracy: 0.9449
Epoch 6/50, Test Accuracy: 0.9485
Epoch 7/50, Test Accuracy: 0.9499
Epoch 8/50, Test Accuracy: 0.9513
Epoch 9/50, Test Accuracy: 0.9519
Epoch 10/50, Test Accuracy: 0.9511
Epoch 11/50, Test Accuracy: 0.9498
Epoch 12/50, Test Accuracy: 0.9529
Epoch 13/50, Test Accuracy: 0.9534
Epoch 14/50, Test Accuracy: 0.9542
Epoch 15/50, Test Accuracy: 0.9506
Epoch 16/50, Test Accuracy: 0.9531
Epoch 17/50, Test Accuracy: 0.9525
Epoch 18/50, Test Accuracy: 0.9531
Epoch 19/50, Test Accuracy: 0.9523
Early stopping at epoch 19
Best accuracy: 0.9542
Model saved to: best_mnist_model.pth


In [3]:
from src.experiments.run_mnist import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00010 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4844.7s

EPS=0.00050 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4843.5s

EPS=0.00100 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4840.8s

EPS=0.01000 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.92 | time=4841.5s


/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(



EPS=0.05000 | clean_acc=0.92 | cert_rate=0.92 | robust_acc(PGD)=0.83 | time=4862.8s


### 7. Final Conclusion

Experiments on MNIST demonstrated a consistent relationship between model capacity, perturbation strength $\epsilon$, and both empirical and certified robustness.

For small values of $\epsilon$, the model maintains high classification stability: clean accuracy, PGD-based robust accuracy, and certified robustness are all close to each other. This indicates that within a small perturbation regime, the decision boundary remains locally stable and both empirical (PGD) and formal (SDP-based) evaluations agree.

As $\epsilon$ increases, a gradual degradation in robustness is observed. In particular, PGD robust accuracy and certified rate decrease, reflecting that larger perturbation sets contain adversarial examples that can cross the decision boundary. This behavior is consistent with theoretical expectations for shallow fully-connected architectures.

A key observation is that certified robustness (SDP-based) and empirical robustness (PGD-based) remain aligned across most tested regimes, which validates the correctness of the implemented certification pipeline and attack procedure.

From the architectural perspective, even a small network can achieve near-perfect clean accuracy on MNIST, but robustness is highly sensitive to representation bottlenecks. Increasing model capacity improves stability margins, but does not eliminate the fundamental trade-off between accuracy and adversarial robustness.

The use of a top-k restricted class setting significantly reduces computational complexity while preserving meaningful robustness comparisons. This allows efficient estimation of both certified and empirical robustness without full-scale combinatorial evaluation over all classes.

**Experiments showed that:**

1. For small values ​​of $\epsilon$, the model exhibits high stability.
2. As $\epsilon$ increases, stability decreases.
3. The PGD and SDP results are consistent.
4. Increasing model power improves both accuracy and stability.
5. Using top-k classes is an effective tradeoff between accuracy and computational complexity.


The obtained results demonstrate consistency between empirical stability (PGD) and formal certification (SDP), which confirms the correctness of the implemented approach.

## CIFAR-10

### 1. Beseline

**Model Setup**

- **Architecture:** 784 → 12 → 10 + ReLU
- **Size of immages:** 16×16×3 (downsampled)
- **Epochs:** 3 
- **Accuracy:** 0.4198

**Attack and verification parameters**

- $\epsilon \in$ {1e-3, 1e-2}
- Samples: 3
- PGD: 20 steps, $\alpha = \epsilon / 10$
- Verification: top-2 classes

**Results:**

- EPS=0.00100 | clean_acc=0.33 | cert_rate=0.33 | robust_acc(PGD)=0.33 | time=738.9s
- EPS=0.01000 | clean_acc=0.33 | cert_rate=0.33 | robust_acc(PGD)=0.33 | time=742.5s

**Conclusion**

- Low model quality → low stability;
- PGD and SDP give identical scores.

This is normal because:

- The model often makes mistakes even at clean levels;
- The stability margin is minimal.

**The baseline model on CIFAR-10 demonstrates low accuracy and, consequently, low stability. The overlap in metrics is explained by the model's limited expressive power.**

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainloader, testloader = get_cifar_loaders()

model = CIFARModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_model(
    model=model,
    trainloader=trainloader,
    testloader=testloader,
    optimizer=optimizer,
    device=device,
    num_epochs=3,
    patience=5,
    save_path="best_cifar_model.pth"
)

Starting training...
Epoch 1/3, Test Accuracy: 0.3857
Epoch 2/3, Test Accuracy: 0.4062
Epoch 3/3, Test Accuracy: 0.4198
Best accuracy: 0.4198
Model saved to: best_cifar_model.pth


In [4]:
from src.experiments.run_cifar import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)



EPS=0.00100 | clean_acc=0.33 | cert_rate=0.33 | robust_acc(PGD)=0.33 | time=738.9s

EPS=0.01000 | clean_acc=0.33 | cert_rate=0.33 | robust_acc(PGD)=0.33 | time=742.5s


AFTER THAT EPOCH 50 with early stop 26: Best accuracy: 0.4673

### 2. Better model

**Model Setup**

- **Architecture:** 784 → 16 → 10 + ReLU
- **Size of immages:** 16×16×3 (downsampled)
- **Epochs:** 50 (early stop at 26th) 
- **Accuracy:** 0.4673

**Attack and verification parameters**

- $\epsilon \in$ {1e-4, 5e-4, 1e-3, 1e-2, 5e-2}
- Samples: 10
- PGD: 25 steps, $\alpha = \epsilon / 10$
- Verification: top-2 classes

**Results:**

- EPS=0.00010 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3509.9s
- EPS=0.00050 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3552.0s
- EPS=0.00100 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3552.8s
- EPS=0.01000 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.40 | time=3297.7s
- EPS=0.05000 | clean_acc=0.50 | cert_rate=0.20 | robust_acc(PGD)=0.20 | time=2520.0s

**Conclusion**

1. Model improvement:
- Accuracy increased from 0.33 → 0.50;
- Cert_rate also increased.

As a result, the model became more robust.

2. Behavior for small $\epsilon$ robust_acc ≥ cert_rate

This is expected because PGD is an approximate method and SDP is a strong guarantee.

3. Behavior for large $\epsilon$ both metrics drop to 0.20

This means:

- The model becomes vulnerable;
- The stability region decreases sharply.

**Increasing the hidden layer dimension leads to an increase in both the accuracy and robustness of the model. The difference between cert_rate and robust_acc reflects the difference between the guaranteed and empirical robustness estimates.**

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

trainloader, testloader = get_cifar_loaders()

model = CIFARModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_model(
    model=model,
    trainloader=trainloader,
    testloader=testloader,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    patience=5,
    save_path="best_cifar_model.pth"
)

Starting training...
Epoch 1/50, Test Accuracy: 0.4080
Epoch 2/50, Test Accuracy: 0.4321
Epoch 3/50, Test Accuracy: 0.4359
Epoch 4/50, Test Accuracy: 0.4467
Epoch 5/50, Test Accuracy: 0.4478
Epoch 6/50, Test Accuracy: 0.4524
Epoch 7/50, Test Accuracy: 0.4515
Epoch 8/50, Test Accuracy: 0.4597
Epoch 9/50, Test Accuracy: 0.4565
Epoch 10/50, Test Accuracy: 0.4584
Epoch 11/50, Test Accuracy: 0.4559
Epoch 12/50, Test Accuracy: 0.4635
Epoch 13/50, Test Accuracy: 0.4613
Epoch 14/50, Test Accuracy: 0.4616
Epoch 15/50, Test Accuracy: 0.4633
Epoch 16/50, Test Accuracy: 0.4617
Epoch 17/50, Test Accuracy: 0.4665
Epoch 18/50, Test Accuracy: 0.4622
Epoch 19/50, Test Accuracy: 0.4590
Epoch 20/50, Test Accuracy: 0.4628
Epoch 21/50, Test Accuracy: 0.4673
Epoch 22/50, Test Accuracy: 0.4652
Epoch 23/50, Test Accuracy: 0.4646
Epoch 24/50, Test Accuracy: 0.4628
Epoch 25/50, Test Accuracy: 0.4647
Epoch 26/50, Test Accuracy: 0.4661
Early stopping at epoch 26
Best accuracy: 0.4673
Model saved to: best_cifar_mo

In [3]:
from src.experiments.run_cifar import run

run()

/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/atoms/affine/reshape.py:68: FutureWarning: 
    You didn't specify the order of the reshape expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(reshape_order_warning, FutureWarning)
/opt/anaconda3/envs/mnist-robust/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(



EPS=0.00010 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3509.9s

EPS=0.00050 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3552.0s

EPS=0.00100 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.50 | time=3552.8s

EPS=0.01000 | clean_acc=0.50 | cert_rate=0.40 | robust_acc(PGD)=0.40 | time=3297.7s

EPS=0.05000 | clean_acc=0.50 | cert_rate=0.20 | robust_acc(PGD)=0.20 | time=2520.0s


### 3. Final Conclusion

Experiments on CIFAR-10 showed that model performance depends significantly on both the architecture and the admissible attack size $\epsilon$.

The baseline model (768 → 12 → 10) exhibits low accuracy (~0.33–0.42) and virtually identical clean accuracy, certified rate, and PGD-robust accuracy values. This indicates the model's limited expressive power and lack of significant robustness.

With increasing hidden layer size (768 → 16 → 10), a steady increase in classification quality (up to ~0.50) and improved robustness are observed. For small $\epsilon$ values, the model maintains a high level of correctness and close certified and empirical robustness values, confirming the consistency of the formal (SDP) and attack (PGD) estimates.

As$\epsilon$ increases, the expected deterioration of all robustness metrics is observed: PGD-robust accuracy and certified rate decrease, reflecting the narrowing of the range of permissible perturbations under which the model maintains classification accuracy.

**Overall, the CIFAR-10 results confirm that:**

- A more capacious architecture improves both accuracy and robustness;
- PGD and SDP yield consistent estimates in the range of small and medium $\epsilon$;
- As $\epsilon$ increases, the model rapidly loses robustness, consistent with theoretical expectations for inherently linear classifiers with shallow depth.


## Conclusion

This study examined the robustness of neural networks on the MNIST and CIFAR-10 datasets using two approaches:

1. Empirical evaluation (PGD)

2. Formal certification (SDP)

**Key results:**

1. For small ε values: 
- The models demonstrate high robustness;
- Attacks are ineffective;
- Certification confirms robustness.

2. As ε increases:
- Obustness decreases predictably;
- PGD begins to find successful attacks;
- SDP detects a decrease in the guaranteed domain.

3. Method consistency:
- PGD and SDP yield similar estimates (SDP <= PGD);
- There are no contradictions between empirical and theoretical data.

4. Influence of architecture increasing the number of neurons:
- Increases accuracy;
- Improves robustness;
- Expands the domain of certifiability.

5. CIFAR-10, as a more complex problem:
- Requires more powerful models;
- Demonstrates more realistic behavior;
- The difference between the metrics becomes more noticeable.

**The obtained results demonstrate that the proposed approach allows for a consistent assessment of the resilience of neural networks using both attacks and formal verification, while the behavior of the models corresponds to the expected theoretical properties.**